Cell 1: Setup & Configuration
This cell initializes the project, checks for available hardware, and sets up a global configuration dictionary for all key parameters.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import os
import cv2
import re
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import GradScaler
from albumentations.pytorch import ToTensorV2
import albumentations as A
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

# Hardware & Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE}")

CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    
    # Heads
    'num_seg_classes': 6,
    'num_det_classes': 80,
    
    # Stereo Geometry
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 48, # 192px / stride 4 = 48
    
    # Training
    'batch_size': 8,
    'lr': 2e-4,
    'num_epochs': 25,
    'num_workers': 4,
    'save_dir': "./checkpoints"
}
os.makedirs(CONFIG['save_dir'], exist_ok=True)

print("✅ Setup complete. Configuration loaded.")


Cell 2: Fused Model Architecture
This cell contains the complete, final architecture for the FusedHexapodModel, including the shared MobileNetV3 backbone and all three specialized heads (Stereo, Segmentation, and Detection).

In [ ]:
# --- Helper Blocks ---
class ConvMean(nn.Module):
    """ Replaces mean() with 1x1 Conv for NPU compatibility """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, 1, bias=False)
        with torch.no_grad():
            self.conv.weight.fill_(1.0 / in_channels)
        self.conv.weight.requires_grad = False
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    """ Standard ResBlock for Refinement """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

# --- Stereo Components ---
class SimpleCorrelation(nn.Module):
    """ Loop-based Correlation + Immediate Reduction (Low Memory) """
    def __init__(self, in_channels, max_disp):
        super().__init__()
        self.D = max_disp
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, 16, 1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True)
        )
        self.reducer = ConvMean(16) 

    def forward(self, left, right):
        l, r = self.reduce(left), self.reduce(right)
        cost_stack = []
        for d in range(self.D):
            if d > 0:
                sim = self.reducer(l[:,:,:,d:] * r[:,:,:,:-d])
                cost_stack.append(F.pad(sim, (d, 0, 0, 0)))
            else:
                cost_stack.append(self.reducer(l * r))
        return torch.cat(cost_stack, dim=1)

class GaussianGating(nn.Module):
    """ Gate = exp(-0.5 * (dist/sigma)^2) """
    def __init__(self, max_disp, gate_range=12):
        super().__init__()
        self.sigma = gate_range / 2.0
        self.disp_coords = nn.Parameter(
            torch.arange(max_disp).float().view(1, max_disp, 1, 1), 
            requires_grad=False
        )

    def forward(self, cost_volume, coarse_disp_s4):
        dist = self.disp_coords - coarse_disp_s4
        return cost_volume * torch.exp(-0.5 * (dist / self.sigma)**2)

class StereoHeadGatedV2(nn.Module):
    def __init__(self, ch_s16, ch_s4, max_disp_s4):
        super().__init__()
        self.D = max_disp_s4
        self.D_coarse = self.D // 4
        
        # Coarse Stage
        self.corr_coarse = SimpleCorrelation(ch_s16, self.D_coarse)
        self.refine_coarse = nn.Sequential(
            nn.Conv2d(self.D_coarse, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32),
            nn.Conv2d(32, self.D_coarse, 3, 1, 1, bias=False)
        )
        self.coarse_sum = nn.Conv2d(self.D_coarse, 1, 1, bias=False) 
        with torch.no_grad():
            self.coarse_sum.weight.data = torch.arange(self.D_coarse).float().view(1, -1, 1, 1)
        self.scale_bias = nn.Parameter(torch.zeros(1, 1, 1, 1))

        # Fine Stage
        self.corr_fine = SimpleCorrelation(ch_s4, self.D)
        self.gating = GaussianGating(max_disp=self.D)
        self.refine_fine = nn.Sequential(
            nn.Conv2d(self.D, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32),
            nn.Conv2d(32, self.D, 3, 1, 1, bias=False) 
        )

    def forward(self, l16, r16, l4, r4):
        c_cost = self.refine_coarse(self.corr_coarse(l16, r16))
        prob_c = F.softmax(c_cost * 2.0, dim=1)
        d_coarse_low = self.coarse_sum(prob_c)
        d_coarse_s4 = (F.interpolate(d_coarse_low, size=l4.shape[-2:], mode='bilinear', align_corners=False) * 4.0) + self.scale_bias 
        
        gated_cost = self.gating(self.corr_fine(l4, r4), d_coarse_s4)
        final_cost = self.refine_fine(gated_cost)
        
        return c_cost, final_cost, d_coarse_s4

# --- Segmentation & Detection Heads ---
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(high_ch, 128, 1, bias=False), nn.Sigmoid()
        )
        self.low_classifier = nn.Conv2d(low_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)

    def forward(self, x_low, x_high):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, size=x_low.shape[-2:], mode='bilinear', align_corners=False)
        return self.low_classifier(x_low) + self.high_classifier(out)

class DecoupledHead(nn.Module):
    def __init__(self, in_ch, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 1, bias=False), nn.BatchNorm2d(in_ch), nn.SiLU(inplace=True)
        )
        self.cls_branch = nn.Sequential(ResBlock(in_ch), ResBlock(in_ch), nn.Conv2d(in_ch, num_classes, 1))
        self.reg_branch = nn.Sequential(ResBlock(in_ch), ResBlock(in_ch), nn.Conv2d(in_ch, 4, 1))

    def forward(self, x):
        x = self.stem(x)
        return self.cls_branch(x), self.reg_branch(x)

class YOLOHead(nn.Module):
    def __init__(self, ch_dims, num_classes):
        super().__init__()
        self.head_s8  = DecoupledHead(ch_dims[1], num_classes)
        self.head_s16 = DecoupledHead(ch_dims[2], num_classes)
        self.head_s32 = DecoupledHead(ch_dims[3], num_classes)

    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# --- Main Fused Model ---
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        bb_ch = self.backbone.feature_info.channels() # [24, 40, 112, 960]
        
        self.stereo_head = StereoHeadGatedV2(ch_s16=bb_ch[2], ch_s4=bb_ch[0], max_disp_s4=config['internal_disp_steps'])
        self.seg_head = LRASPPHead(low_ch=bb_ch[0], high_ch=bb_ch[2], num_classes=config['num_seg_classes'])
        self.yolo_head = YOLOHead(ch_dims=bb_ch, num_classes=config['num_det_classes'])

    def forward(self, left, right=None):
        # Student model expects grayscale, repeats to 3 channels for backbone
        x = left.repeat(1, 3, 1, 1) 
        fl = self.backbone(x)
        
        # Stereo Branch (only if right image provided)
        stereo_preds = None
        if right is not None:
            xr = right.repeat(1, 3, 1, 1)
            fr = self.backbone(xr)
            stereo_preds = self.stereo_head(fl[2], fr[2], fl[0], fr[0])
            
        # Other Heads
        seg_preds = self.seg_head(fl[0], fl[2])
        det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        
        # Return in a consistent order for the loss function
        return stereo_preds, seg_preds, det_preds

model = FusedHexapodModel(CONFIG).to(DEVICE)
print("✅ Fused model architecture cell is ready.")


Cell 3: Unified Loss Function
This cell contains the corrected and unified loss function. It properly calculates the stereo loss on the refined output and combines it with segmentation and detection losses.

In [ ]:
class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes=80):
        super().__init__()
        self.num_classes = num_classes
        
    def _get_targets(self, targets_list, cls_preds, reg_preds):
        batch_size, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        cls_targets = torch.zeros_like(cls_preds)
        reg_targets = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((batch_size, H, W), device=device, dtype=torch.bool)
        
        for b, t in enumerate(targets_list):
            if t is None or t.numel() == 0: continue
            
            t = t[t[:, 1:5].sum(dim=1) > 0] # Filter out empty boxes
            if t.numel() == 0: continue

            gt_cls = t[:, 0].long()
            gt_x = (t[:, 1] * W).long()
            gt_y = (t[:, 2] * H).long()
            gt_w = t[:, 3] * W
            gt_h = t[:, 4] * H
            
            # Clamp to grid boundaries
            gt_x = torch.clamp(gt_x, 0, W - 1)
            gt_y = torch.clamp(gt_y, 0, H - 1)

            obj_mask[b, gt_y, gt_x] = True
            cls_targets[b, gt_cls, gt_y, gt_x] = 1.0
            reg_targets[b, 0, gt_y, gt_x] = gt_w / 2.0
            reg_targets[b, 1, gt_y, gt_x] = gt_h / 2.0
            reg_targets[b, 2, gt_y, gt_x] = gt_w / 2.0
            reg_targets[b, 3, gt_y, gt_x] = gt_h / 2.0
                
        return cls_targets, reg_targets, obj_mask

    def forward(self, preds, targets):
        # This loss calculates for one scale (e.g., S32)
        cls_p, reg_p = preds
        cls_t, reg_t, mask = self._get_targets(targets, cls_p, reg_p)
        
        loss_cls = F.binary_cross_entropy_with_logits(cls_p, cls_t, reduction='sum') / (mask.sum() + 1e-6)
        
        if mask.any():
            mask_reg = mask.unsqueeze(1).repeat(1, 4, 1, 1)
            loss_reg = F.smooth_l1_loss(reg_p[mask_reg], reg_t[mask_reg], beta=1.0, reduction='sum') / (mask.sum() + 1e-6)
        else:
            loss_reg = torch.tensor(0.0, device=cls_p.device)
            
        return loss_cls + loss_reg

class FusedHexapodLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.yolo_s8_loss = SimpleYOLOLoss(config['num_det_classes'])
        self.yolo_s16_loss = SimpleYOLOLoss(config['num_det_classes'])
        self.yolo_s32_loss = SimpleYOLOLoss(config['num_det_classes'])
        self.seg_ce_loss = nn.CrossEntropyLoss(ignore_index=255)
        
        self.w_stereo = 1.0
        self.w_seg = 1.0
        self.w_kd = 0.5
        self.w_yolo = 1.0
        self.temp_kd = 4.0

    def soft_argmax(self, cost_volume):
        """Converts a cost volume to a disparity map."""
        if cost_volume is None: return None
        probs = F.softmax(cost_volume, dim=1)
        indices = torch.arange(cost_volume.shape[1], device=cost_volume.device, dtype=torch.float32).view(1, -1, 1, 1)
        return torch.sum(probs * indices, dim=1, keepdim=True)

    def forward(self, preds, targets, teacher_preds=None):
        stereo_preds, seg_preds, det_preds = preds
        logs = {}
        total_loss = 0

        # --- 1. Stereo Loss (Corrected) ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            
            if mask.sum() > 0:
                # *** CRITICAL FIX: Use the FINE cost volume (index 1) ***
                fine_cost_volume = stereo_preds[1]
                pred_disp_s4 = self.soft_argmax(fine_cost_volume)
                pred_disp_full = F.interpolate(pred_disp_s4, size=gt_disp.shape[-2:], mode='bilinear', align_corners=False) * CONFIG['backbone_stride']
                
                l_stereo = F.smooth_l1_loss(pred_disp_full[mask], gt_disp[mask])
                total_loss += self.w_stereo * l_stereo
                logs['stereo'] = l_stereo.item()

        # --- 2. Segmentation Loss ---
        if seg_preds is not None and targets.get('seg') is not None:
            gt_seg = targets['seg']
            seg_preds_up = F.interpolate(seg_preds, size=gt_seg.shape[-2:], mode='bilinear', align_corners=False)
            
            # GT Loss
            l_seg = self.seg_ce_loss(seg_preds_up, gt_seg)
            total_loss += self.w_seg * l_seg
            logs['seg_gt'] = l_seg.item()
            
            # Distillation Loss
            if teacher_preds is not None and 'seg' in teacher_preds:
                t_logits = teacher_preds['seg']
                p = F.softmax(t_logits / self.temp_kd, dim=1)
                q = F.log_softmax(seg_preds_up / self.temp_kd, dim=1)
                kl_div = F.kl_div(q, p, reduction='batchmean')
                l_kd = kl_div * (self.temp_kd ** 2)
                total_loss += self.w_kd * l_kd
                logs['seg_kd'] = l_kd.item()

        # --- 3. YOLO Loss (Multi-scale) ---
        if det_preds is not None and targets.get('det') is not None:
            gt_det = targets['det']
            l_yolo8  = self.yolo_s8_loss(det_preds[0], gt_det)
            l_yolo16 = self.yolo_s16_loss(det_preds[1], gt_det)
            l_yolo32 = self.yolo_s32_loss(det_preds[2], gt_det)
            
            l_yolo = l_yolo8 + l_yolo16 + l_yolo32
            total_loss += self.w_yolo * l_yolo
            logs['yolo'] = l_yolo.item()
            
        return total_loss, logs

criterion = FusedHexapodLoss(CONFIG).to(DEVICE)
print("✅ Unified loss function cell is ready.")


Cell 4: Data Parsing Helpers
This cell contains the necessary helper functions to scan the respective dataset directories (FlyingThings3D, TartanAir, and COCO) and create a unified list of file paths and labels. The RealFusedDataset class in the next cell will use these functions.

In [ ]:
import glob
from pycocotools.coco import COCO

# Note: The 'pycocotools' library is required for this cell: pip install pycocotools
def parse_ft3d(root):
    """ Scans the FlyingThings3D directory for stereo pairs and disparity maps. """
    print("   Scanning FlyingThings3D...")
    samples = []
    # Adjust paths if your structure differs
    img_dir_l = os.path.join(root, 'frames_cleanpass/TRAIN')
    disp_dir_l = os.path.join(root, 'disparity/TRAIN')
    
    if not os.path.exists(img_dir_l): return []
    
    # Iterate through subfolders (e.g., A, B, C)
    for scene_folder in sorted(os.listdir(img_dir_l)):
        left_files = sorted(glob.glob(os.path.join(img_dir_l, scene_folder, 'left', '*.png')))
        
        for l_path in left_files:
            filename = os.path.basename(l_path)
            r_path = l_path.replace('/left/', '/right/')
            # The disparity files are .pfm
            d_path = os.path.join(disp_dir_l, scene_folder, 'left', filename.replace('.png', '.pfm'))
            
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({'type':'stereo','source':'ft3d','l':l_path,'r':r_path,'d':d_path,'s':None,'b':None})

    print(f"   -> Found {len(samples)} FT3D pairs.")
    return samples

def parse_tartan(root):
    """ Scans the TartanAir directory for stereo pairs, depth, and optional segmentation. """
    print("   Scanning TartanAir...")
    samples = []
    # Find all 'image_left' folders, which indicates a valid sequence
    left_folders = glob.glob(os.path.join(root, '**', 'image_left'), recursive=True)
    
    for l_folder in left_folders:
        parent = os.path.dirname(l_folder)
        for f in os.listdir(l_folder):
            if not f.endswith('.png'): continue
            
            l_path = os.path.join(l_folder, f)
            r_path = os.path.join(parent, 'image_right', f.replace('_left', '_right'))
            d_path = os.path.join(parent, 'depth_left', f.replace('.png', '_depth.npy'))
            s_path = os.path.join(parent, 'seg_left', f.replace('.png', '_seg.npy'))
            
            # Ensure critical files exist
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({
                    'type': 'stereo', 'source': 'tartan',
                    'l': l_path, 'r': r_path, 'd': d_path, 
                    's': s_path if os.path.exists(s_path) else None, 
                    'b': None
                })
    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco(root):
    """ Scans the COCO directory for images and bounding box annotations. """
    print("   Scanning COCO 2017...")
    samples = []
    ann_file = os.path.join(root, 'annotations', 'instances_train2017.json')
    img_dir = os.path.join(root, 'train2017')
    if not os.path.exists(ann_file): return []
    
    coco = COCO(ann_file)
    for iid in coco.getImgIds():
        img_info = coco.loadImgs(iid)[0]
        path = os.path.join(img_dir, img_info['file_name'])
        if not os.path.exists(path): continue
        
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, iscrowd=False))
        boxes = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            H, W = img_info['height'], img_info['width']
            # Convert to YOLO format [class, x_center, y_center, width, height]
            boxes.append([ann['category_id'] - 1, (x + w/2)/W, (y + h/2)/H, w/W, h/H])
            
        if boxes:
            samples.append({'type':'mono','source':'coco','l':path,'b':np.array(boxes, dtype=np.float32),'r':None,'d':None,'s':None})
            
    print(f"   -> Found {len(samples)} COCO samples.")
    return samples

print("✅ Data parsing helper functions are ready.")


Cell 5: Data Loading & Augmentation
This cell defines the RealFusedDataset class. It uses the parsers from the previous cell to build a master file list and then applies appropriate augmentations for training. It also includes the corrected .pfm file reader.

In [ ]:
def read_pfm_fixed(file_path):
    """ Reads a .pfm file and returns a numpy array. Includes scaling. """
    with open(file_path, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        endian = '<' if scale < 0 else '>'
        scale = abs(scale)
        
        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data) # PFM files are stored upside down
    return (data * scale).copy()

class RealFusedDataset(Dataset):
    def __init__(self, roots, mode='train', img_size=(480, 640)):
        self.img_h, self.img_w = img_size
        self.samples = []
        
        if 'ft3d' in roots and os.path.exists(roots['ft3d']): self.samples.extend(parse_ft3d(roots['ft3d']))
        if 'tartan' in roots and os.path.exists(roots['tartan']): self.samples.extend(parse_tartan(roots['tartan']))
        if 'coco' in roots and os.path.exists(roots['coco']): self.samples.extend(parse_coco(roots['coco']))
        
        print(f"📊 Total Samples Found: {len(self.samples)}")
        
        # Augmentations for the main (left) image and its targets
        self.transform_main = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
        
        # Simpler augmentation for the right image (no targets)
        self.transform_right = A.Compose([
            A.Resize(height=self.img_h, width=self.img_w),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_l_rgb = cv2.cvtColor(cv2.imread(s['l']), cv2.COLOR_BGR2RGB)
        h, w, _ = img_l_rgb.shape

        # Load targets based on sample type
        disp = read_pfm_fixed(s['d']) if s.get('d') and s['d'].endswith('.pfm') else (np.load(s['d']) if s.get('d') else np.full((h, w), -1.0, dtype=np.float32))
        seg = np.load(s['s']) if s.get('s') else np.full((h, w), 255, dtype=np.uint8)
        boxes, class_labels = (list(zip(*[(b[1:], b[0]) for b in s['b']])) if s.get('b') is not None and len(s['b']) > 0 else ([], []))
        
        # Apply main transform to left image and all its targets
        transformed = self.transform_main(image=img_l_rgb, masks=[disp, seg], bboxes=boxes, class_labels=class_labels)
        teacher_input = transformed['image']
        t_disp = transformed['masks'][0].unsqueeze(0)
        t_seg = transformed['masks'][1].long()
        
        # Convert grayscale for the robot model input
        robot_input_l = (teacher_input[0]*0.299 + teacher_input[1]*0.587 + teacher_input[2]*0.114).unsqueeze(0)

        # Handle detection boxes
        t_det = torch.tensor([[cl] + list(b) for cl, b in zip(transformed['class_labels'], transformed['bboxes'])]) if transformed['bboxes'] else torch.zeros((0, 5))

        # Handle right image
        if s['type'] == 'stereo':
            img_r_rgb = cv2.cvtColor(cv2.imread(s['r']), cv2.COLOR_BGR2RGB)
            robot_input_r = (self.transform_right(image=img_r_rgb)['image'][0]*0.299 + ...).unsqueeze(0)
        else: # For mono images, duplicate left as right
            robot_input_r = robot_input_l.clone()

        return {'left': robot_input_l, 'right': robot_input_r, 'teacher': teacher_input, 'disp': t_disp, 'seg': t_seg, 'det': t_det}

print("✅ RealFusedDataset is ready.")


Cell 6: The Final Training Loop
This cell sets up and executes the main training process. It includes the crucial hexapod_collate function to handle mixed-data batches, initializes the data loaders, optimizer, and scheduler, and contains the complete training and validation logic with logging to TensorBoard.

In [ ]:
# --- Segmentation Teacher Model ---
class SegmentationTeacher(nn.Module):
    """
    A wrapper for a pre-trained Hugging Face semantic segmentation model that
    maps the output to a custom robotics class space.
    """
    def __init__(self, device='cuda'):
        super().__init__()
        model_name = "shariqfarooq/deeplabv3-resnet50-nyu-v2-40-class"
        
        # 1. Load the model and its processor
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForSemanticSegmentation.from_pretrained(model_name)
        
        # 2. Freeze the teacher model
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
        
        # 3. Create the Class Mapping Tensor
        # Define our 6 target robotics classes as integers
        self.FLOOR = 0
        self.WALL = 1
        self.STATIC_OBJ = 2
        self.DYNAMIC_OBJ = 3
        self.STAIRS = 4
        self.VOID = 5  # A general "ignore" or "background" class
        
        # Create a mapping from the 40 NYUv2 class indices to our 6 classes
        # This list must have 40 elements.
        nyu_to_robotics_map = [
            # 0: wall -> WALL
            self.WALL,
            # 1: floor -> FLOOR
            self.FLOOR,
            # 2: cabinet -> STATIC_OBJ
            self.STATIC_OBJ,
            # 3: bed -> STATIC_OBJ
            self.STATIC_OBJ,
            # 4: chair -> STATIC_OBJ
            self.STATIC_OBJ,
            # 5: sofa -> STATIC_OBJ
            self.STATIC_OBJ,
            # 6: table -> STATIC_OBJ
            self.STATIC_OBJ,
            # 7: door -> WALL
            self.WALL,
            # 8: window -> WALL
            self.WALL,
            # 9: bookshelf -> STATIC_OBJ
            self.STATIC_OBJ,
            # 10: picture -> WALL
            self.WALL,
            # 11: counter -> STATIC_OBJ
            self.STATIC_OBJ,
            # 12: blinds -> WALL
            self.WALL,
            # 13: desk -> STATIC_OBJ
            self.STATIC_OBJ,
            # 14: shelves -> STATIC_OBJ
            self.STATIC_OBJ,
            # 15: curtain -> WALL
            self.WALL,
            # 16: dresser -> STATIC_OBJ
            self.STATIC_OBJ,
            # 17: pillow -> DYNAMIC_OBJ (often movable)
            self.DYNAMIC_OBJ,
            # 18: mirror -> WALL
            self.WALL,
            # 19: floor mat -> FLOOR
            self.FLOOR,
            # 20: clothes -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 21: ceiling -> VOID (ignore)
            self.VOID,
            # 22: books -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 23: refridgerator -> STATIC_OBJ
            self.STATIC_OBJ,
            # 24: television -> STATIC_OBJ
            self.STATIC_OBJ,
            # 25: paper -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 26: towel -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 27: shower curtain -> VOID
            self.VOID,
            # 28: box -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 29: whiteboard -> WALL
            self.WALL,
            # 30: person -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 31: night stand -> STATIC_OBJ
            self.STATIC_OBJ,
            # 32: toilet -> STATIC_OBJ
            self.STATIC_OBJ,
            # 33: sink -> STATIC_OBJ
            self.STATIC_OBJ,
            # 34: lamp -> STATIC_OBJ
            self.STATIC_OBJ,
            # 35: bathtub -> STATIC_OBJ
            self.STATIC_OBJ,
            # 36: bag -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ,
            # 37: otherstructure -> VOID
            self.VOID,
            # 38: otherfurniture -> STATIC_OBJ
            self.STATIC_OBJ,
            # 39: otherprop -> DYNAMIC_OBJ
            self.DYNAMIC_OBJ
        ]
        
        # Register as a buffer to ensure it moves to the correct device with .to()
        self.register_buffer('nyu_map', torch.tensor(nyu_to_robotics_map, dtype=torch.long))

        print(f"✅ Teacher model '{model_name}' loaded and class map created.")

    def to(self, device):
        self.model.to(device)
        return super().to(device)

    def forward(self, x):
        with torch.no_grad():
            # 1. Run inference with the base model
            inputs = self.processor(images=x, return_tensors="pt", do_rescale=False).to(self.model.device)
            nyu_logits = self.model(**inputs).logits
            
            # 2. Get the predicted NYU class index for each pixel
            # This is the "hard distillation" step.
            nyu_class_indices = torch.argmax(nyu_logits, dim=1) # Shape: [B, H, W]
            
            # 3. Use the mapping tensor to translate to our 6 robotics classes
            robotics_class_indices = self.nyu_map[nyu_class_indices] # Shape: [B, H, W]
            
            # 4. Convert the mapped indices back to a one-hot logit format
            # This creates a [B, 6, H, W] tensor for the loss function.
            # We scale by a large number to simulate strong "logits" for the chosen class.
            robotics_logits = F.one_hot(robotics_class_indices, num_classes=6).permute(0, 3, 1, 2).float() * 10.0
            
            # 5. Upsample the final logits to match the input size
            upsampled_logits = F.interpolate(
                robotics_logits,
                size=x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            
        return upsampled_logits

# --- Verification Step ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Initialize the teacher and move it to the GPU/CPU
teacher = SegmentationTeacher().to(DEVICE)

# 2. Create a dummy batch of RGB images
# Shape: [Batch Size, Channels, Height, Width]
dummy_rgb_input = torch.rand(2, 3, 480, 640).to(DEVICE)

# 3. Perform a forward pass
teacher_logits = teacher(dummy_rgb_input)

# 4. Check the output shape
print("\n--- Teacher Integration Test ---")
print(f"Input Shape:  {dummy_rgb_input.shape}")
print(f"Output Shape: {teacher_logits.shape}")

# The output shape should be [Batch Size, 40, Height, Width]
# 40 is the number of classes in the NYUv2 dataset.
if teacher_logits.shape == (2, 40, 480, 640):
    print("✅ SUCCESS: The teacher model produced logits with the correct shape.")
else:
    print("❌ FAILURE: The output shape is incorrect. Please review the code.")


# --- Custom Collate Function ---
def hexapod_collate(batch):
    """ Custom collate to handle variable numbers of detection boxes per image. """
    keys = batch[0].keys()
    collated = {k: [d[k] for d in batch] for k in keys}
    
    # Stack tensors that have a fixed size
    for k in ['left', 'right', 'teacher', 'disp', 'seg']:
        collated[k] = torch.stack(collated[k])
    # 'det' remains a list of tensors
    return collated

# --- Data Setup ---
DATA_ROOTS = {
    'coco':   '../datasets/coco',
    'ft3d':   '../datasets/FlyingThings3D',
    'tartan': '../datasets/TartanAir'
}
full_dataset = RealFusedDataset(DATA_ROOTS, img_size=(CONFIG['img_height'], CONFIG['img_width']))
train_size = int(0.95 * len(full_dataset))
train_ds, val_ds = random_split(full_dataset, [train_size, len(full_dataset) - train_size])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True, collate_fn=hexapod_collate)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True, collate_fn=hexapod_collate)

# --- Training Setup ---
teacher_seg = SegmentationTeacher(CONFIG['num_seg_classes']).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CONFIG['lr'], epochs=CONFIG['num_epochs'], steps_per_epoch=len(train_loader))
scaler = GradScaler()
writer = SummaryWriter("./logs")

def train_one_epoch(epoch_idx):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_idx+1}/{CONFIG['num_epochs']}")
    
    for i, batch in enumerate(pbar):
        # Move data to device
        l, r, t_img = batch['left'].to(DEVICE), batch['right'].to(DEVICE), batch['teacher'].to(DEVICE)
        targets = {'disp': batch['disp'].to(DEVICE), 'seg': batch['seg'].to(DEVICE), 'det': [t.to(DEVICE) for t in batch['det']]}

        # Teacher pass (for distillation)
        with torch.no_grad():
            teacher_preds = {'seg': teacher_seg(t_img)}

        # Student forward/backward pass
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE, dtype=torch.float16):
            preds = model(l, r)
            loss, logs = criterion(preds, targets, teacher_preds)
        
        if torch.isnan(loss) or torch.isinf(loss): continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Logging
        if i % 50 == 0:
            step = epoch_idx * len(train_loader) + i
            writer.add_scalar('Loss/Total', loss.item(), step)
            for k, v in logs.items(): writer.add_scalar(f'Loss/{k}', v, step)
        pbar.set_postfix({'loss': f"{loss.item():.3f}", 'lr': f"{scheduler.get_last_lr()[0]:.2e}"})

# --- Main Execution ---
print("🚀 Starting training...")
for epoch in range(CONFIG['num_epochs']):
    train_one_epoch(epoch)
    # Add validation loop here if needed
    torch.save(model.state_dict(), os.path.join(CONFIG['save_dir'], f"hexapod_epoch_{epoch+1}.pth"))
print("🏁 Training complete.")
